# Scrapping

In [3]:
import requests
from bs4 import BeautifulSoup
import time
import os
import re
import json


BASE_URL = "https://www.canada.ca"
# START_PAGE = "https://www.canada.ca/en/immigration-refugees-citizenship/services/immigrate-canada.html"
START_PAGE = "https://www.canada.ca/en/immigration-refugees-citizenship/services/immigrate-canada/express-entry/check-score/crs-criteria.html"
MAX_PAGES = 10000
DATA_DIR = "data_1"

In [4]:
def clean_text(text):
    text = re.sub(r'\r\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()


def detect_category(title):
    title_lower = title.lower()
    if "express entry" in title_lower:
        return "Express Entry"
    elif "provincial nominee" in title_lower:
        return "PNP"
    elif "family sponsorship" in title_lower:
        return "Family Sponsorship"
    else:
        return "General"

In [ ]:
from pprint import pprint

In [ ]:
# response = requests.get(START_PAGE, timeout=10)
# soup = BeautifulSoup(response.text, "lxml")
# pprint(soup.find("title"))
# pprint(soup)
# for tag in soup(["header", "footer", "nav", "script", "style"]):
#     tag.decompose()
# pprint(soup.find("main").get_text(separator="\n", strip=True))

In [7]:
import requests
from bs4 import BeautifulSoup, Tag
from urllib.parse import urlparse
from datetime import datetime

In [5]:
# URL = "https://www.canada.ca/en/immigration-refugees-citizenship/services/immigrate-canada/express-entry/check-score/crs-criteria.html"
URL = "https://www.canada.ca/en/immigration-refugees-citizenship/services/refugees.html"


def clean_html(soup: BeautifulSoup):
    # Remove unwanted tags
    for tag in soup(["script", "style", "nav", "header", "footer", "noscript"]):
        tag.decompose()

    # Remove "On this page" section
    for h2 in soup.find_all("h2"):
        if "On this page" in h2.get_text():
            parent = h2.find_parent()
            if parent:
                parent.decompose()

    return soup

In [8]:
def table_to_markdown(table: Tag) -> str:
    rows = []
    headers = []

    # Extract headers
    header_row = table.find("thead")
    if header_row:
        headers = [
            th.get_text(strip=True).replace("\xa0", " ")
            for th in header_row.find_all("th")
        ]
        rows.append("| " + " | ".join(headers) + " |")
        rows.append("| " + " | ".join(["---"] * len(headers)) + " |")

    # Extract body rows
    for tr in table.find_all("tr"):
        cols = tr.find_all(["td", "th"])
        row = [col.get_text(strip=True).replace("\xa0", " ") for col in cols]
        if row:
            rows.append("| " + " | ".join(row) + " |")

    return "\n".join(rows)

In [11]:
def extract_documents(url, html_string=None):
    # response = requests.get(url)
    # soup = BeautifulSoup(response.text, "lxml")
    soup = BeautifulSoup(html_string, 'html.parser')
    
    meta_tags = extract_meta(soup)
    breadcrumb = extract_breadcrumb(soup)
    
    program = breadcrumb[-2] if len(breadcrumb) >= 2 else None
    topic = breadcrumb[-1] if breadcrumb else None
    
    page_title = soup.title.get_text(strip=True)
    parsed_url = urlparse(url)

    base_meta_data = {
        "url": url,
        "path": parsed_url.path,

        "date": meta_tags.get("dcterms.modified"),
        "issued_date": meta_tags.get("dcterms.issued"),
        "language": meta_tags.get("dcterms.language"),

        "program": program,
        "topic": topic,
        "tags": meta_tags.get("dcterms.subject"),
        "metadata_subject": meta_tags.get("dcterms.subject"),

        "page_title": page_title,
        "metadata_title": meta_tags.get("dcterms.title"),

        "section_title": None,
        "description": meta_tags.get("description"),
        "keywords": meta_tags.get("keywords"),

        "archived": 'ARCHIVED'.lower() in page_title.lower()
    }

    main = soup.find("main")
    main = clean_html(main)

    documents = []

    h1 = main.find("h1")
    intro_content = []
    capture = False
    
    for element in main.descendants:
    
        if isinstance(element, Tag):
    
            # When we hit H1 → start capturing
            if element.name == "h1":
                capture = True
                continue
    
            # Stop when first H2 appears
            if element.name == "h2":
                break
    
            if capture:
                if element.name == "table":
                    intro_content.append(table_to_markdown(element))
    
                elif element.name in ["p", "li"]:
                    text = element.get_text(" ", strip=True)
                    if text:
                        intro_content.append(text)
    if intro_content:
        documents.append({
            "content": "\n".join(intro_content).strip(),
            "metadata": build_metadata(base_meta_data, h1.get_text(strip=True))
        })
        

    # ---- STEP 2: Capture H2/H3 Sections ----
    current_section = None
    current_content = []

    for element in main.find_all(["h2", "h3", "h4", "p", "li", "table"]):

        if element.name in ["h2", "h3", "h4"]:
            # Save previous section
            if current_section and current_content:
                documents.append({
                    "content": "\n".join(current_content).strip(),
                    "metadata": build_metadata(base_meta_data, current_section)
                })

            current_section = element.get_text(strip=True)
            current_content = []

        elif element.name == "table":
            current_content.append(table_to_markdown(element))

        elif element.name in ["p", "li"]:
            text = element.get_text(" ", strip=True)
            if text:
                current_content.append(text)

    # Save last section
    if current_section and current_content:
        documents.append({
            "content": "\n".join(current_content).strip(),
            "metadata": build_metadata(base_meta_data, current_section)
        })

    return documents

In [12]:
def extract_meta(soup):
    meta = {}

    for tag in soup.find_all("meta"):
        name = tag.get("name")
        content = tag.get("content")

        if name and content:
            meta[name.lower()] = content

    return meta

def extract_breadcrumb(soup):
    breadcrumb = []
    nav = soup.find("nav", {"aria-label": "Breadcrumb"})
    
    if nav:
        for li in nav.find_all("li"):
            text = li.get_text(strip=True)
            if text:
                breadcrumb.append(text)

    return breadcrumb

    
def build_metadata(m_dict, section_title):
    m = m_dict.copy()
    m['section_title'] = section_title
    return m

In [19]:
with open(f'data/immigrate-canada_1.json', "r") as f:
    json_data = json.load(f)

In [26]:
docs = extract_documents(json_data['url'], json_data['xml'])

In [27]:


# Example preview
for d in docs:
    print("\n--- SECTION ---")
    # print("Date:", d["metadata"]["date"])
    # print("Path:", d["metadata"]["path"])
    # print("Program:", d["metadata"]["program"])
    # print("Tag:", d["metadata"]["tag"])
    # print("Topic:", d["metadata"]["topic"])
    # print("Pate Title:", d["metadata"]["page_title"])
    print("Section Title:", d["metadata"]["section_title"])

    print("CONTENT PREVIEW:", d["content"])


--- SECTION ---
Section Title: Live in Canada permanently
CONTENT PREVIEW: Explore permanent residence programs to move to Canada, such as Express Entry, family sponsorship and regional programs.

--- SECTION ---
Section Title: Express Entry
CONTENT PREVIEW: Skilled workers, tradespeople

--- SECTION ---
Section Title: Caregivers
CONTENT PREVIEW: Home care workers

--- SECTION ---
Section Title: Medical doctors
CONTENT PREVIEW: For physicians

--- SECTION ---
Section Title: Get a work permit
CONTENT PREVIEW: Work in Canada for a few years with a work permit

--- SECTION ---
Section Title: Explore all immigration programs
CONTENT PREVIEW: Answer a few questions to get a list of programs to explore

--- SECTION ---
Section Title: Francophone immigration
CONTENT PREVIEW: Live, work or study in French in a Francophone community outside Quebec

--- SECTION ---
Section Title: From:
CONTENT PREVIEW: Immigration, Refugees and Citizenship Canada
Immigration and Refugee Board of Canada


In [24]:
def extract_documents(url, html_string=None):
    # response = requests.get(url)
    # soup = BeautifulSoup(response.text, "lxml")
    soup = BeautifulSoup(html_string, 'html.parser')
    
    meta_tags = extract_meta(soup)
    breadcrumb = extract_breadcrumb(soup)
    
    program = breadcrumb[-2] if len(breadcrumb) >= 2 else None
    topic = breadcrumb[-1] if breadcrumb else None
    
    page_title = soup.title.get_text(strip=True)
    parsed_url = urlparse(url)

    base_meta_data = {
        "url": url,
        "path": parsed_url.path,

        "date": meta_tags.get("dcterms.modified"),
        "issued_date": meta_tags.get("dcterms.issued"),
        "language": meta_tags.get("dcterms.language"),

        "program": program,
        "topic": topic,
        "tags": meta_tags.get("dcterms.subject"),
        "metadata_subject": meta_tags.get("dcterms.subject"),

        "page_title": page_title,
        "metadata_title": meta_tags.get("dcterms.title"),

        "section_title": None,
        "description": meta_tags.get("description"),
        "keywords": meta_tags.get("keywords"),

        "archived": 'ARCHIVED'.lower() in page_title.lower()
    }

    main = soup.find("main")
    main = clean_html(main)

    documents = []

    h1 = main.find("h1")
    intro_content = []
    capture = False
    
    for element in main.descendants:
    
        if isinstance(element, Tag):
    
            # When we hit H1 → start capturing
            if element.name == "h1":
                capture = True
                continue
    
            # Stop when first H2 appears
            if element.name == "h2":
                break
    
            if capture:
                if element.name == "table":
                    intro_content.append(table_to_markdown(element))
    
                elif element.name in ["p", "li"]:
                    text = element.get_text(" ", strip=True)
                    if text:
                        intro_content.append(text)
    if intro_content:
        documents.append({
            "content": "\n".join(intro_content).strip(),
            "metadata": build_metadata(base_meta_data, h1.get_text(strip=True))
        })
        

    # ---- STEP 2: Capture H2/H3 Sections ----
    current_section = None
    current_content = []

    for element in main.find_all(["h2", "h3", "h4", "p", "li", "table"]):

        if element.name in ["h2", "h3", "h4"]:
            # Save previous section
            if current_section and current_content:
                documents.append({
                    "content": "\n".join(current_content).strip(),
                    "metadata": build_metadata(base_meta_data, current_section)
                })

            current_section = element.get_text(strip=True)
            current_content = []

        elif element.name == "table":
            current_content.append(table_to_markdown(element))

        elif element.name in ["p", "li"]:
            text = element.get_text(" ", strip=True)
            if text:
                current_content.append(text)

    # Save last section
    if current_section and current_content:
        documents.append({
            "content": "\n".join(current_content).strip(),
            "metadata": build_metadata(base_meta_data, current_section)
        })

    return documents

# Filtering


In [16]:
import pandas as pd
import json
import statistics
from transformers import AutoTokenizer
from pathlib import Path

/Users/ankit/Projects/learn_llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
files[0]

'canadian-citizenship_45.json'

In [ ]:
df = pd.DataFrame(pages)
df.head()

In [ ]:
df['archived'] = df['content'].apply(lambda x: 'ARCHIVED'.lower() in x.lower())

In [ ]:
df['archived'].value_counts()

In [ ]:
df['date'] = df['content'].apply(lambda x: x[-10:])

In [ ]:
df['char_counts'] = df['content'].apply(lambda x: len(x))
df['word_counts'] = df['content'].apply(lambda x: len(x.split()))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")
df['token_counts'] = df['content'].apply(lambda x: len(tokenizer.encode(x)))

In [ ]:
df[['char_counts', 'word_counts', 'token_counts']].describe().astype(int)

In [ ]:
df.to_parquet('data.parquet')

# Embbeding 

In [ ]:
import pandas as pd
df = pd.read_parquet('data.parquet')

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=300
)

documents = []

for i, (url, title, category, content, _, archived, date, _, _, _) in df.iterrows():
    chunks = splitter.split_text(content[:-23])

    for chunk in chunks:
        doc = Document(
            page_content=chunk,
            metadata={
                "title": title,
                "url": url,
                "category": category,
                "archived": archived,                
                "date": date,
                
            }
        )
        documents.append(doc)
print(f"Total Pages: {len(df)}")
print(f"Total chunked documents: {len(documents)}")

In [ ]:
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")
persist_directory = "vector_db/ircc"

In [ ]:
# vector_store = Chroma.from_documents(
#     documents,
#     embedding=embeddings,
#     persist_directory=persist_directory
# )
# vector_store.persist()
# print(f"Vector DB persisted to '{persist_directory}' directory.")

In [ ]:
from tqdm import tqdm
from langchain_community.vectorstores import Chroma

batch_size = 1000

for i in tqdm(range(0, len(documents), batch_size), desc="Indexing batches"):
    batch_docs = documents[i:i + batch_size]

    Chroma.from_documents(
        batch_docs,
        embedding=embeddings,
        persist_directory=persist_directory
    )

In [ ]:
# print("Loading existing vector DB...")

# vector_store = Chroma(
#     persist_directory=persist_directory,
#     embedding_function=embeddings
# )

# print("Vector DB loaded successfully.")